In [ ]:
import re
import time
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Load the Naukri Jobs Dataset

In [ ]:
df = pd.read_csv("/content/marketing_sample_for_naukri_com-jobs__20190701_20190830__30k_data.csv")
print("Shape:", df.shape)
df.head(3)

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(subset=["Job Title", "Key Skills", "Functional Area"], inplace=True)

In [ ]:
df.duplicated().sum()

In [ ]:
df.dropna(subset=["Key Skills", "Job Title", "Functional Area"], inplace=True)
print("Shape after dropping missing values:", df.shape)

## 2. Clean Text

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"<.*?>", " ", text)          # strip any stray HTML
    text = text.replace("|", " ")                 # Key Skills are pipe-separated
    text = re.sub(r"[^a-z0-9\s]", " ", text)      # keep letters/numbers
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
# Combine job title + key skills into a single text field (our "review" equivalent)
df["text"] = df["Job Title"].astype(str) + " " + df["Key Skills"].astype(str)
df["clean_text"] = df["text"].apply(clean_text)

## 3. Build the Target: Functional Area (Top 15 + Other)

In [ ]:
# The raw "Functional Area" column has 70+ categories with a long tail.
# Keep the top 15 and bucket everything else into "Other" (matches the site's own convention).
TOP_N = 15
top_classes = df["Functional Area"].value_counts().head(TOP_N).index.tolist()
df["target"] = df["Functional Area"].where(df["Functional Area"].isin(top_classes), "Other")

le = LabelEncoder()
df["label"] = le.fit_transform(df["target"])
print("Classes:", list(le.classes_))

## 4. Train/Test Split

In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["clean_text"], df["label"],
    test_size=0.2, random_state=RANDOM_STATE, stratify=df["label"]
)

## Prepare Sequences for the Neural Network

### Tokenization
Convert each cleaned job posting into a sequence of integers, where each integer represents a word's rank in the vocabulary (fit **only** on the training set).

### Padding
Neural networks need fixed-length input. We pad/truncate every sequence to `max_len=40` (job titles + skills are much shorter than movie reviews).

In [ ]:
tokenizer = Tokenizer(num_words=8000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_test_seq = tokenizer.texts_to_sequences(X_test_text)

X_train_pad = pad_sequences(X_train_seq, maxlen=40, padding="post", truncating="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=40, padding="post", truncating="post")

y_train_arr = y_train.values
y_test_arr = y_test.values

## 5. Build & Train the LSTM Model

In [ ]:
lstm_model = Sequential([
    Embedding(input_dim=8000, output_dim=64, mask_zero=True),

    LSTM(64),

    Dropout(0.3),

    Dense(32, activation="relu"),

    Dense(len(le.classes_), activation="softmax")
])

lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True
)

history_lstm = lstm_model.fit(
    X_train_pad,
    y_train_arr,
    validation_split=0.1,
    epochs=15,
    batch_size=256,
    callbacks=[early_stop]
)

## 6. Evaluate

In [1]:
y_pred_lstm = np.argmax(lstm_model.predict(X_test_pad), axis=1)

print("Test accuracy:", accuracy_score(y_test_arr, y_pred_lstm))
print(classification_report(y_test_arr, y_pred_lstm, target_names=le.classes_))

Test accuracy: 0.6926

                                                              precision    recall  f1-score   support

        Accounts , Finance , Tax , Company Secretary , Audit       0.73      0.81      0.77       265
                                    Engineering Design , R&D       0.52      0.29      0.38        92
      Financial Services , Banking , Investments , Insurance       0.38      0.35      0.36       133
                      HR , Recruitment , Administration , IR       0.78      0.80      0.79       271
         IT Software - Application Programming , Maintenance       0.70      0.85      0.77      1441
                                     IT Software - ERP , CRM       0.42      0.05      0.10        92
                                         IT Software - Other       0.00      0.00      0.00        83
                                  IT Software - QA & Testing       0.53      0.60      0.57        81
      ITES , BPO , KPO , LPO , Customer Service , Operatio

## 7. Save Artifacts

In [ ]:
lstm_model.save("lstm_model.h5")

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)
with open("label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

In [ ]:
from google.colab import files

files.download("lstm_model.h5")
files.download("tokenizer.pkl")
files.download("label_encoder.pkl")